# 2장 후보 모델의 공식 평가 비교

## 이번 질문

앞 노트북의 검증 숫자는 공식 승인이 아닙니다. 이 노트북은 모델을 다시 조정하지 않고 준비된 공식 평가 자료에서 기준 모델, Candidate A, Candidate B의 품질 지표와 판단을 비교합니다. 후보는 이미 `profiles.yaml`에서 고른 상태입니다. Candidate A를 보류하고 Candidate B를 승인한 이유를 봉인 평가 숫자로 설명하는 것이 목표입니다.


## 먼저 예상

Candidate A와 Candidate B 가운데 어느 후보가 승인됐을지 먼저 고릅니다. PR-AUC 하나가 아니라 재현율 하한, 정밀도와 FN 감소 조건 중 어떤 항목이 결론을 바꿀지도 예상합니다.

## 실행과 관측

입력은 `docs/evidence/model-v2/canonical-benchmark.json`입니다. 이 파일은 공식 평가를 한 번만 연 결과이며, 이 노트북은 값을 바꾸지 않고 읽기만 합니다.

In [ ]:
from pathlib import Path

import json
import pandas as pd

# 1. 저장소 루트를 찾는다.
ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

# 2. 공식 평가 JSON을 연다. 이 노트북은 값을 고치지 않는다.
EVIDENCE_PATH = ROOT / "docs/evidence/model-v2/canonical-benchmark.json"
evidence = json.loads(EVIDENCE_PATH.read_text(encoding="utf-8"))
assert evidence["sealed_test"]["status"] == "evaluated_once"
assert evidence["deployment_allowed"] is True
pd.DataFrame(
    {"값": [EVIDENCE_PATH.relative_to(ROOT).as_posix(), evidence["sealed_test"]["status"]]},
    index=["공식 평가 파일", "평가 상태"],
)


In [ ]:
# 1. JSON에 적힌 후보 판단을 먼저 본다.
# JSON에 적힌 후보 판단을 먼저 본다. HOLD / APPROVE 이유를 다음 셀에서 숫자로 푼다.
pd.DataFrame(evidence["decisions"]).set_index("profile")[["decision", "checks"]]


### 1. 평가 전에 고정한 후보 읽기

슬라이드의 model selection은 이미 `profiles.yaml`에 끝난 상태다. 이 셀은 세 후보가 무엇인지만 읽고, 여기서 다시 고르거나 학습하지 않는다.


In [ ]:
import yaml

# 1. 이미 고른 세 후보와 배포 기준을 연다.
POLICY_PATH = ROOT / "configs/qa-v2.yaml"
PROFILES_PATH = ROOT / "configs/model-v2/profiles.yaml"
BOOTSTRAP_PATH = ROOT / "docs/evidence/model-v2/model-bootstrap.json"
policy = yaml.safe_load(POLICY_PATH.read_text(encoding="utf-8"))
profiles = yaml.safe_load(PROFILES_PATH.read_text(encoding="utf-8"))
bootstrap = json.loads(BOOTSTRAP_PATH.read_text(encoding="utf-8"))

# 2. 고른 것: 모델 종류, 불균형 가중, 예측 임계값.
rows = []
for profile in profiles["profiles"]:
    params = profile.get("params", {})
    rows.append(
        {
            "profile": profile["name"],
            "kind": profile["kind"],
            "class_weight": params.get("class_weight"),
            "threshold": profile["threshold"],
            "C": params.get("C"),
            "n_estimators": params.get("n_estimators"),
        }
    )
pd.DataFrame(rows).set_index("profile")


### 2. 공통 공식 평가 지표 비교

같은 공식 평가에서 기준 모델, A, B의 정밀도·재현율·미탐을 한 표로 본다.


In [ ]:
# 1. 공식 JSON의 지표와 판단을 한 표로 붙인다.
# 공식 JSON의 지표와 판단을 한 표로 붙인다.
decisions = {}
for item in evidence["decisions"]:
    decisions[item["profile"]] = item["decision"]

rows = []
for profile in evidence["profiles"]:
    metrics = profile["metrics"]
    name = profile["profile"]
    rows.append(
        {
            "profile": name,
            "threshold": profile["threshold"],
            "pr_auc": metrics["pr_auc"],
            "precision": metrics["precision"],
            "recall": metrics["recall"],
            "false_negative": metrics["false_negative"],
            "decision": decisions.get(name, "REFERENCE"),
        }
    )
comparison = pd.DataFrame(rows).set_index("profile")
comparison.round(4)


### 3. 배포 기준을 수치로 다시 확인

각 행은 하나의 보호 질문이다. `passed`가 False인 항목이 Candidate A를 보류한 이유이다.


In [ ]:
# 1. 후보마다 같은 다섯 보호 질문을 다시 계산한다.
baseline = comparison.loc[policy["baseline_profile"]]
rules = policy["rules"]
required_recall = rules["minimum_recall"] + rules["recall_safety_margin"]

# 공식 JSON에 적힌 재현율 불확실성 하한
recall_lower = {}
for item in evidence["profiles"]:
    recall_lower[item["profile"]] = item["bootstrap_recall_lower"]

# 후보마다 같은 다섯 질문을 한다: 실제값 >= 필요한 값 이면 통과.
rows = []
for profile in (policy["candidate_a_profile"], policy["candidate_b_profile"]):
    actual = comparison.loc[profile]
    rows.append(
        {
            "profile": profile,
            "check": "recall_guardrail",
            "actual": actual["recall"],
            "required": required_recall,
        }
    )
    rows.append(
        {
            "profile": profile,
            "check": "recall_uncertainty",
            "actual": recall_lower[profile],
            "required": rules["minimum_recall_bootstrap_lower"],
        }
    )
    rows.append(
        {
            "profile": profile,
            "check": "precision_floor",
            "actual": actual["precision"],
            "required": rules["minimum_precision"],
        }
    )
    rows.append(
        {
            "profile": profile,
            "check": "pr_auc_vs_baseline",
            "actual": actual["pr_auc"] - baseline["pr_auc"],
            "required": rules["minimum_pr_auc_delta_vs_baseline"],
        }
    )
    rows.append(
        {
            "profile": profile,
            "check": "false_negative_reduction",
            "actual": baseline["false_negative"] - actual["false_negative"],
            "required": rules["minimum_false_negative_reduction"],
        }
    )

policy_comparison = pd.DataFrame(rows)
policy_comparison["passed"] = (
    policy_comparison["actual"] >= policy_comparison["required"]
)
policy_comparison.round(4)


### 4. 배포 기준 통과 여부 확인

PR-AUC 하나만으로 승인하지 않습니다. 재현율의 불확실성 하한, 정밀도 최솟값, FN 감소 조건을 함께 확인합니다.

In [ ]:
# 1. 공식 JSON이 이미 매긴 통과/실패를 확인한다.
# 공식 JSON이 이미 매긴 통과/실패이다. 위 표와 같은지 확인한다.
checks = pd.DataFrame(
    {item["profile"]: item["checks"] for item in evidence["decisions"]}
).T
checks


## 해석과 기록

Candidate A와 Candidate B의 판단을 통과하지 못한 보호 기준과 함께 누적 기록에 옮깁니다. 모델 승인과 대상 환경의 배포 상태는 아직 합치지 않습니다.

## 결과 점검

In [ ]:
assert decisions == {"candidate-a": "HOLD", "candidate-b": "APPROVE"}
assert comparison.loc["candidate-b", "recall"] > comparison.loc["baseline", "recall"]
assert comparison.loc["candidate-b", "false_negative"] < comparison.loc["baseline", "false_negative"]
assert not checks.loc["candidate-a"].all()
assert checks.loc["candidate-b"].all()
print("Canonical model evidence checks passed.")

## 다음 확인

`02_trace_model_lineage.ipynb`에서 DVC revision, 공식 MLflow Run, model/metadata SHA-256과 release manifest를 Candidate B 하나로 연결합니다. 이어서 새 학생 Run을 실행해 같은 모델 프로필의 설정값, 검증 지표, dataset input과 생성 파일을 MLflow에서 조회합니다. 이 노트북에서는 특성이나 임계값을 변경하지 않습니다.